In [ ]:
import rasterio
from rasterio.features import shapes
import numpy as np
from shapely.geometry import shape, MultiPolygon, mapping, Point
import fiona
from fiona.crs import from_epsg
from scipy.ndimage import label
import matplotlib.pyplot as plt
import reverse_geocoder as rg
import pycountry
from rasterio.plot import show
import os


# Step 1: Read the raster file
input_raster_path = r'D:\RIDA-Docker\work\polygon\T47QME_20210318_2\T47QME_20210318T035539_B1210.tif'
output_shapefile_path = r'D:\RIDA-Docker\work\polygon\Output\burn_condition.shp'

with rasterio.open(input_raster_path) as src:
    raster_data = src.read(1)  # Read the first band
    transform = src.transform  # Get the affine transform

# Step 2: Extract features based on burn condition values
burn_condition = (raster_data == 1).astype(np.uint8)

# Label connected components
labeled_array, num_features = label(burn_condition)

# Step 3: Convert labeled features to polygons
shapes_generator = shapes(labeled_array, transform=transform)

polygons = []
for geom, value in shapes_generator:
    if value > 0:  # Only take the features corresponding to burn condition
        polygons.append(shape(geom))

# Combine polygons into a single MultiPolygon
multi_polygon = MultiPolygon(polygons)

# Step 4: Save the polygons to a shapefile
schema = {
    'geometry': 'MultiPolygon',
    'properties': {'id': 'int'},
}

# Ensure output directory exists
os.makedirs(os.path.dirname(output_shapefile_path), exist_ok=True)

with fiona.open(output_shapefile_path, 'w', 'ESRI Shapefile', schema=schema, crs=from_epsg(4326)) as shp:
    shp.write({
        'geometry': mapping(multi_polygon),
        'properties': {'id': 1},
    })

print(f"Shapefile saved to {output_shapefile_path}")

# Reverse geocode to get city and province
def get_location_info(latitude, longitude):
    coordinates = (latitude, longitude)
    result = rg.search(coordinates)
    if result:
        location = result[0]
        city = location['name']
        province = location['admin1']
        country_code = location['cc']
        country = pycountry.countries.get(alpha_2=country_code).name
        return city, province, country
    return None, None, None

# Step 5: Print properties and plot the output
fig, ax = plt.subplots(figsize=(10, 10))

# Plot the original raster data
ax.imshow(raster_data, cmap='gray', extent=(transform[2], transform[2] + transform[0] * raster_data.shape[1], transform[5] + transform[4] * raster_data.shape[0], transform[5]))
ax.set_title("Burn Condition Raster and Polygons")

# Plot the polygons and print their properties
for polygon in polygons:
    x, y = polygon.exterior.xy
    ax.plot(x, y, color='red', linewidth=2)

    # Get centroid of the polygon for reverse geocoding
    centroid = polygon.centroid
    latitude, longitude = centroid.y, centroid.x
    city, province, country = get_location_info(latitude, longitude)

    print(f"Polygon centroid: ({latitude}, {longitude})")
    print(f"City: {city}, Province: {province}, Country: {country}")

plt.show()
